In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd "/content/drive/MyDrive/Masters's Thesis/TM Project/"

/content/drive/MyDrive/Masters's Thesis/TM Project


In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
# !pip install scikit-optimize

In [5]:
# from skopt import gp_minimize
# from skopt import space

In [28]:
import devices
import models
import samplers
import trainer
import utils

import torch
import numpy as np
import matplotlib.pyplot as plt

from scipy.io import loadmat

In [43]:
# parameters
lambd = 1

# wave number and normalized spacing
k_0 = 2 * np.pi / lambd
dx = (lambd / 50) * k_0

# normalized simulation lengths
substrate_length = (6 * lambd) * k_0
core_length = (2 * lambd) * k_0
cladding_length = (6 * lambd) * k_0

# refractive indices
n_substrate = 2
n_core = 2.1
n_cladding = 1.9

In [44]:
memory_type = torch.device('cpu')
dtype = torch.float64
torch.manual_seed(42)
num_modes = 2

In [45]:
base_model = models.discontinuity_capturing_network(1, 256, num_modes, 1, 'RBF')
# base_model = models.vanilla_network(1, 256, num_modes, 1, 'RBF')
optimzer = torch.optim.Adamax(base_model.parameters(), lr=1e-2)
# scheduler = torch.optim.lr_scheduler.MultiStepLR(optimzer, milestones=[1000, 2000, 3000], gamma=0.2)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimzer, milestones=[2000, 3000, 4000], gamma=0.2)
print('number of trainable parameters', sum([p.numel() for p in base_model.parameters() if p.requires_grad]))

number of trainable parameters 1538


In [46]:
device = devices.SlabWaveguideWithPINNs(n_substrate, n_core, n_cladding, lambd, core_length, substrate_length, cladding_length, 'TE', base_model, True)

In [47]:
train_agent = trainer.Trainer(dtype, device, memory_type, [samplers.uniform_grid], optimzer, scheduler, dx)

In [48]:
# loss_history, rayleigh_history, mode_errors_list = train_agent.train(10001, num_modes, weights=[1, 1, 1, 1, 1, 1], verbose=1, sample_new=False)
loss_history, rayleigh_history, mode_errors_list = train_agent.train(10001, num_modes, weights=[10, 0.5, 2, 1, 1, 1], verbose=1, sample_new=False)

num PDE samples 699
num rayleigh samples 699
iteration 1 loss1 is 0.06167, loss2 is 3.73257, loss3 is -5.77846, loss4 is 0.00848, loss5 is 0.073, loss6 is 0.31136
mode errors
**********
NN vs analytic
relative error for mode 0 is 79.5388%
**********
**********
NN vs analytic
relative error for mode 1 is 77.44142%
**********
iteration 1001 loss1 is 0.00315, loss2 is 0.02483, loss3 is -6.48766, loss4 is 0.0028, loss5 is 0.00072, loss6 is 0.01544
mode errors
**********
NN vs analytic
relative error for mode 0 is 4.85102%
**********
**********
NN vs analytic
relative error for mode 1 is 3.91317%
**********
iteration 2001 loss1 is 0.00237, loss2 is 0.03226, loss3 is -6.4881, loss4 is 0.00515, loss5 is 0.00091, loss6 is 0.00808
mode errors
**********
NN vs analytic
relative error for mode 0 is 2.89228%
**********
**********
NN vs analytic
relative error for mode 1 is 3.52341%
**********
iteration 3001 loss1 is 0.00072, loss2 is 0.00015, loss3 is -6.48896, loss4 is 0.0, loss5 is 0.0002, loss6

In [49]:
# reference_device = devices.SlabWaveguide(n_substrate, n_core, n_cladding, lambd, core_length, substrate_length, cladding_length, 'TM')

In [50]:
fd_device = devices.SlabWaveguideWithFD(n_substrate, n_core, n_cladding, lambd, core_length, substrate_length, cladding_length, 'TE', dx)
_, x, u = fd_device.evaluate(num_modes)
utils.compare_against_analytic(fd_device, x, torch.from_numpy(u))

mode errors
**********
NN vs analytic
relative error for mode 0 is 0.64695%
**********
**********
NN vs analytic
relative error for mode 1 is 1.0214%
**********


[0.6469508676180307, 1.0213973438056343]

In [ ]:
x_tensor = torch.from_numpy(x)
x_tensor.requires_grad = True
x_augmented = torch.cat((x_tensor.reshape(-1, 1), device.refractive_index(torch.from_numpy(x).reshape(-1, 1), format='torch')**2), dim=-1)

In [ ]:
_**(1/2), fd_device.evaluate_analytical(x, 1)[1], device.calculate_eigen_value(x_augmented, 0)**(1/2)

In [ ]:
mode = fd_device.evaluate_analytical(x, 1)[1]
mode = mode.reshape(-1)
delta_mode = mode[1:] - mode[:-1]
derivative_mode = delta_mode / dx
discontinuities_indices = device.get_discontinuities_index(torch.from_numpy(x))
print(discontinuities_indices)
print((n_substrate**2)/(n_core**2), (n_core**2)/(n_cladding**2))
print(derivative_mode[395:405]/derivative_mode[396:406])
print(derivative_mode[295:305]/derivative_mode[296:306])

In [ ]:
_**(1/2), fd_device.evaluate_analytical(x, 1)[1], device.calculate_eigen_value(x_augmented, 0)**(1/2)

In [ ]:
analytical_mode_matlab = torch.from_numpy(loadmat('./Matlab Data/mode.m')['Fy']).reshape(-1)
evaluation_points_matlab = loadmat('./Matlab Data/evaluation_points.m')['x_out']
analytical_mode_matlab = analytical_mode_matlab / torch.linalg.norm(analytical_mode_matlab, dim=0)

In [ ]:
delta_mode = analytical_mode_matlab[1:] - analytical_mode_matlab[:-1]
derivative_mode = delta_mode / dx
discontinuities_indices = device.get_discontinuities_index(torch.from_numpy(x))
print(discontinuities_indices)
print((n_substrate**2)/(n_core**2), (n_core**2)/(n_cladding**2))
print(derivative_mode[395:405]/derivative_mode[396:406])
print(derivative_mode[295:305]/derivative_mode[296:306])

In [ ]:
analytical_mode_python = device.evaluate_analytical(evaluation_points_matlab * k_0 - core_length/2, 1)[1].reshape(-1)

In [ ]:
plt.plot(analytical_mode_matlab)
plt.plot(analytical_mode_python)

In [ ]:
FD_output = fd_device.evaluate(1)

plt.figure(figsize=(10, 10))

plt.plot(x, device.evaluate_analytical(x, 1)[1].reshape(-1));
plt.plot(x, NN_output, color='red')
plt.plot(x, -FD_output[2])
plt.vlines(core_length/2, 0, 0.1, linestyle='dashed');
plt.vlines(-core_length/2, 0, 0.1, linestyle='dashed');

In [ ]:
def optimize(params):
    torch.manual_seed(42)
    base_model = models.discontinuity_capturing_network(1, 128, 1, 1, 'RBF')
    optimzer = torch.optim.Adamax(base_model.parameters(), lr=1e-2)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimzer, milestones=[1000, 2000, 3000], gamma=0.2)
    device = devices.SlabWaveguideWithPINNs(n_substrate, n_core, n_cladding, lambd, core_length, substrate_length, cladding_length, 'TM', base_model)
    train_agent = trainer.Trainer(dtype, device, memory_type, [samplers.uniform_grid], optimzer, scheduler, dx)
    return train_agent.train(5001, 1, weights=params, verbose=1, sample_new=False)[2][0][-1]

In [ ]:
param_space = [
    space.Real(1e0, 1e1, prior='log-uniform'),
    space.Real(1e-1, 1e1, prior='log-uniform'),
    space.Real(1e-1, 1e1, prior='log-uniform'),
    space.Real(1e-1, 1e1, prior='log-uniform'),
    space.Real(1e-1, 1e1, prior='log-uniform'),
    space.Real(1e-1, 1e1, prior='log-uniform'),
]

In [ ]:
gp_minimize(
    optimize,
    dimensions=param_space,
    n_calls=25,
    n_random_starts=15,
    verbose=10
)

In [ ]:
#         loss.append(torch.sum(torch.square(grad_right[i]*epsilon(discontinuities[i] + np.finfo(np.float32).eps) - grad_left[i]*epsilon(discontinuities[i] - np.finfo(np.float32).eps))))